In [1]:
ratpup           <- read.table('data/rat_pup.dat', header=TRUE)
ratpup$sex       <- as.factor(ratpup$sex)
ratpup$litter    <- as.factor(ratpup$litter)
ratpup$treatment <- as.factor(ratpup$treatment)

# Building the Model I


In our second step, we start by building a model for a *single* dependency structure. With clustered data, the entities that create dependencies are the clusters. Remember, these are the *boundaries of independence*. Independence lives between the levels of the cluster and dependence lives within each level of the cluster. As such, a *single dependency structure* can be taken to mean a *single cluster*. So, we start by writing the model for *one cluster*. In doing so, we start by imagining that our data was only collected from a single litter of pups. 

We can look at this in `R` by subsetting the data to just those values from the first litter

In [2]:
litter.1 <- subset(ratpup, litter=='1')
print(litter.1)

   pup_id weight    sex litter litsize treatment
1       1   6.60   Male      1      12   Control
2       2   7.40   Male      1      12   Control
3       3   7.15   Male      1      12   Control
4       4   7.24   Male      1      12   Control
5       5   7.10   Male      1      12   Control
6       6   6.04   Male      1      12   Control
7       7   6.98   Male      1      12   Control
8       8   7.05   Male      1      12   Control
9       9   6.95 Female      1      12   Control
10     10   6.29 Female      1      12   Control
11     11   6.77 Female      1      12   Control
12     12   6.57 Female      1      12   Control


As we can see, `litter`, `litsize` and `treatment` are all *constant* at the level of a single litter and so cannot be used as predictors. This make sense because these either index the data as a whole (`litter`) or are manipulated *across* litters (`litsize`,`treatment`). So, if we only have *one* litter, these are irrelevant. We could even go as far as removing these from `litter.1`. If you are working with very complex datasets, this is something you may want to do to help clarify the model you are building. However, for this example, we will leave the data as-is.

Based on the above, the only useable variables we have at the level of a single litter are `weight` and `sex`. Because there are no repeated measurements, `pup_id` simply indexes the rows of the data frame and is not of much use. So, based on all this, the only model we can build at the level of a single litter is one that models `weights` as a function of the `sex` of the pup. This is not wholly unreasonable, as we expect male pups to be heavier than female pups. So our starting point is the normal linear model

$$
y_{ij} = \mu + \alpha_{j} + \eta_{ij},
$$

where $y_{ij}$ is the value of `weight` for pup $i$ of `sex` $j$, $\mu$ is the average weight of all pups in this litter, $\alpha_{j}$ is the effect of `sex` at level $j$ and $\eta_{ij}$ is the error. This is effectively a one-way ANOVA model of `sex` within a single litter.

We can run this model in `R` to check that we have a model that is possible with a single litter. This is not wholly necessary, but can be a useful exercise when building these models up slowly. It is also useful because we can see the model formula for *one litter*, which will become relevant a little later on.

In [3]:
lm.litter.1 <- lm(weight ~ 1 + sex, data=litter.1)
summary(lm.litter.1)


Call:
lm(formula = weight ~ 1 + sex, data = litter.1)

Residuals:
    Min      1Q  Median      3Q     Max 
-0.9050 -0.1425  0.1150  0.2275  0.4550 

Coefficients:
            Estimate Std. Error t value Pr(>|t|)    
(Intercept)   6.6450     0.1969  33.749 1.23e-11 ***
sexMale       0.3000     0.2411   1.244    0.242    
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 0.3938 on 10 degrees of freedom
Multiple R-squared:  0.134,	Adjusted R-squared:  0.04743 
F-statistic: 1.548 on 1 and 10 DF,  p-value: 0.2418


As expected, `R` has implemented the estimability constraint that $\alpha_{1} = 0$, so $\hat{\mu}$ is now the average value of `weight` for female pups and $\hat{\alpha}_{2}$ is the mean difference between male and female pups. As we can see, in this litter $\hat{\alpha}_{2} = 0.3$, so there is a small increase in `weight` for male pups over female pups. More importantly, we can see that this model *is* possible and is not constrained by the data.

[^sextextbook-foot]: It is interesting to note that, in their description of this analysis, [West, Welch & Galecki (2022)](https://www.taylorfrancis.com/books/mono/10.1201/9781003181064/linear-mixed-models-brady-west-kathleen-welch-andrzej-galecki?context=ubx&refId=4dbda113-7f59-4ba9-a710-897ef536cfee) make the decision to fix `sex` to a constant across litters, under the belief that there is a universal effect of `sex` on weight that is not influenced by individual litters.